# Building an MCP Proxy (Code Review Agent)

Wrap the HTTP server (from `http-server/`) with MCP so AI assistants can call **two tools**:

1. **get_pull_request_files** – get files changed in a pull request (diffs and stats)  
2. **create_pr_review** – add a review comment on a PR (COMMENT, APPROVE, or REQUEST_CHANGES)

**Transport:** Streamable HTTP – one endpoint (`/` or `/mcp`) supports **GET** (SSE stream) and **POST** (JSON-RPC in, response as **SSE** with one event). Notifications (e.g. `initialized`) get **202 Accepted** with no body.

```
AI Assistant  ──MCP (Streamable HTTP / SSE)──►  MCP Proxy  ──HTTP──►  Flask Server  ──HTTP──►  GitHub API
```

**Prerequisite:** Start the HTTP server first:
```bash
cd ../http-server && python app.py
```

## Step 1: Define the two tool schemas

MCP tools: **get_pull_request_files** (owner, repo, pull_number) and **create_pr_review** (owner, repo, pull_number, event, body).

In [ ]:
TOOLS = [
    {
        "name": "get_pull_request_files",
        "description": "Get the list of files changed in a pull request with their diffs and change statistics.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "owner": {"type": "string", "description": "Repository owner"},
                "repo": {"type": "string", "description": "Repository name"},
                "pull_number": {"type": "integer", "description": "Pull request number"},
                "per_page": {"type": "integer", "description": "Results per page", "default": 30},
                "page": {"type": "integer", "description": "Page number", "default": 1},
            },
            "required": ["owner", "repo", "pull_number"],
        },
    },
    {
        "name": "create_pr_review",
        "description": "Create a review comment on a pull request. Use event COMMENT to add a comment, or APPROVE / REQUEST_CHANGES.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "owner": {"type": "string", "description": "Repository owner"},
                "repo": {"type": "string", "description": "Repository name"},
                "pull_number": {"type": "integer", "description": "Pull request number"},
                "event": {"type": "string", "description": "COMMENT, APPROVE, or REQUEST_CHANGES"},
                "body": {"type": "string", "description": "Review comment text"},
            },
            "required": ["owner", "repo", "pull_number", "event"],
        },
    },
]

print(f"Defined {len(TOOLS)} tools:", [t["name"] for t in TOOLS])

## Step 2: Create the proxy layer

Map each MCP tool call to the HTTP backend: GET .../pulls/{pull_number}/files and POST .../pulls/{pull_number}/reviews.

In [ ]:
import json
import urllib.request
import urllib.error

BACKEND_URL = "http://localhost:5000"

def http_get(path):
    with urllib.request.urlopen(f"{BACKEND_URL}{path}") as resp:
        return json.loads(resp.read())

def http_post(path, data):
    req = urllib.request.Request(
        f"{BACKEND_URL}{path}",
        data=json.dumps(data).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req) as resp:
        return json.loads(resp.read())

def call_tool(name, args):
    if name == "get_pull_request_files":
        path = f"/repos/{args['owner']}/{args['repo']}/pulls/{args['pull_number']}/files"
        params = f"per_page={args.get('per_page', 30)}&page={args.get('page', 1)}"
        data = http_get(f"{path}?{params}")
        if isinstance(data, list):
            out = [{"filename": f.get("filename"), "additions": f.get("additions"), "deletions": f.get("deletions")} for f in data[:5]]
            return json.dumps(out, indent=2)
        return json.dumps(data)

    if name == "create_pr_review":
        path = f"/repos/{args['owner']}/{args['repo']}/pulls/{args['pull_number']}/reviews"
        payload = {"event": args["event"]}
        if args.get("body"):
            payload["body"] = args["body"]
        data = http_post(path, payload)
        return json.dumps(data)

    return json.dumps({"error": "Unknown tool"})

## Step 3: Test the proxy

Call **get_pull_request_files** (no token needed for public repos). **create_pr_review** requires the HTTP server to have `GITHUB_TOKEN` and write access to the repo.

In [ ]:
# Make sure http-server is running (cd ../http-server && python app.py)!

# 1. Get pull request files (public repo)
result = call_tool("get_pull_request_files", {"owner": "octocat", "repo": "hello-world", "pull_number": 1})
print("get_pull_request_files:", result[:500] + "..." if len(result) > 500 else result)

# 2. Add review comment – uncomment and set owner/repo/pull_number to a PR you can write to
# result = call_tool("create_pr_review", {"owner": "YOU", "repo": "REPO", "pull_number": 1, "event": "COMMENT", "body": "Tutorial: MCP proxy test"})
# print("create_pr_review:", result)

## Step 4: JSON-RPC handlers

In `server.py`: requests (with `id`) get a JSON-RPC response sent as **one SSE event** (`Content-Type: text/event-stream`, then `data: <json>\n\n`). Notifications (no `id`) get **202 Accepted** with no body. Below we just return the response dict; the real server wraps it in SSE.

In [ ]:
def handle_request(req):
    method = req.get("method")
    req_id = req.get("id")
    
    # Notifications (no id) → in server.py we return 202 Accepted, no body
    if req_id is None:
        return None  # server sends 202, no response body
    
    if method == "initialize":
        return {"jsonrpc": "2.0", "id": req_id, "result": {"protocolVersion": "2024-11-05", "capabilities": {"tools": {"listChanged": True}}, "serverInfo": {"name": "github-pr-review-proxy", "version": "1.0.0"}}}
    
    if method == "tools/list":
        return {"jsonrpc": "2.0", "id": req_id, "result": {"tools": TOOLS}}
    
    if method == "tools/call":
        params = req.get("params", {})
        result = call_tool(params.get("name"), params.get("arguments", {}))
        return {"jsonrpc": "2.0", "id": req_id, "result": {"content": [{"type": "text", "text": result}]}}
    
    return {"jsonrpc": "2.0", "id": req_id, "error": {"code": -32601, "message": "Method not found"}}

# Test (same as server: request gets a result; server would send this as SSE)
print(json.dumps(handle_request({"jsonrpc": "2.0", "id": 1, "method": "tools/list"}), indent=2))

## Step 5: Run the server

See `server.py` and `proxy.py` for the full implementation.

- **Streamable HTTP:** One endpoint (`/` or `/mcp`). **POST** = JSON-RPC body; response is **SSE** (one event, JSON-RPC in `data:`). **GET** = SSE stream (minimal event then close). Notifications → 202 Accepted.
- No OPTIONS or origin checks in the demo.

```bash
# Terminal 1: Start HTTP backend
cd ../http-server && python app.py

# Terminal 2: Start MCP proxy
python server.py
```

**List tools (curl).** Response is SSE, e.g. `data: {"jsonrpc":"2.0","id":1,"result":{...}}\n\n`:
```bash
curl -s -X POST http://localhost:8080/mcp \
  -H "Content-Type: application/json" \
  -d '{"jsonrpc":"2.0","id":1,"method":"tools/list"}'
```

**GET** (SSE stream):
```bash
curl -s -N http://localhost:8080/mcp
```

**Call get_pull_request_files** (response is one SSE event):
```bash
curl -s -X POST http://localhost:8080/mcp \
  -H "Content-Type: application/json" \
  -d '{"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"get_pull_request_files","arguments":{"owner":"octocat","repo":"hello-world","pull_number":1}}}'
```

**Call create_pr_review** (needs GITHUB_TOKEN on http-server):
```bash
curl -s -X POST http://localhost:8080/mcp \
  -H "Content-Type: application/json" \
  -d '{"jsonrpc":"2.0","id":3,"method":"tools/call","params":{"name":"create_pr_review","arguments":{"owner":"OWNER","repo":"REPO","pull_number":1,"event":"COMMENT","body":"Review comment"}}}'
```